# `region` 02: noteworthy single-feature findings

**Purpose:** identify and discuss supported points that stand out after the
standard `category` breakdown. Target relationships here are
exploratory and must be rechecked after the split is frozen.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'region'
feature_metadata = {'order': 12, 'name': 'region', 'audit_type': 'category', 'role': 'candidate', 'disposition': 'retain as interpretable geographic back-off', 'finding': 'All 21 levels are covered and functional rates differ substantially by region.', 'decision': 'Retain and add an LGA/region-grouped validation sensitivity check.', 'risk': 'Random validation can reward geographic memorisation.', 'related': [{'feature': 'region_code', 'reason': 'The named and coded fields overlap but are not simple duplicates.'}, {'feature': 'lga', 'reason': 'Each LGA maps to one region in the supplied data.'}, {'feature': 'basin', 'reason': 'Hydrological basins cross administrative regions.'}, {'feature': 'longitude', 'reason': 'Coordinates should broadly agree with named region.'}, {'feature': 'latitude', 'reason': 'Coordinates should broadly agree with named region.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for region.


## Supported target evidence


In [2]:
sentinel_tokens = []
target_profile = categorical_target_profile(
    training_data,
    feature,
    minimum_support=100,
    sentinel_tokens=sentinel_tokens,
)
display(target_profile.head(20))

supported = target_profile.loc[target_profile["meets support threshold"]].copy()
non_functional_column = "non functional (%)"
if non_functional_column in supported:
    display(
        supported.sort_values(non_functional_column, ascending=False)
        .head(12)[["rows", non_functional_column]]
    )


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
region,,,,,
iringa,5294,True,78.22,2.32,19.46
shinyanga,4982,True,55.98,12.75,31.27
mbeya,4639,True,49.99,10.86,39.15
kilimanjaro,4379,True,60.29,7.35,32.36
morogoro,4006,True,52.90,7.49,39.62
arusha,3350,True,68.48,5.22,26.30
kagera,3316,True,52.08,9.17,38.75
mwanza,3102,True,48.42,5.90,45.68
kigoma,2816,True,48.40,21.41,30.18


status_group,rows,non functional (%)
region,,
lindi,1546,64.23
mtwara,1730,62.43
tabora,1959,54.42
rukwa,1808,53.43
mara,1969,51.96
mwanza,3102,45.68
singida,2093,45.58
dodoma,2201,44.66
dar es salaam,805,42.36


## Observation

All 21 levels are covered and functional rates differ substantially by region.

## Interpretation

The supported single-feature patterns make this field worth the stated
treatment, but they do not prove causation or independent predictive value.
High-cardinality and geographic fields are especially vulnerable to
memorisation under a random split.

## Provisional decision

Retain and add an LGA/region-grouped validation sensitivity check.

**Risk to carry forward:** Random validation can reward geographic memorisation.


In [3]:
decision_record = pd.DataFrame([{
    "feature": feature,
    "role": feature_metadata["role"],
    "disposition": feature_metadata["disposition"],
    "finding": feature_metadata["finding"],
    "decision": feature_metadata["decision"],
    "risk": feature_metadata["risk"],
}])
display(decision_record.set_index("feature"))


,role,disposition,finding,decision,risk
feature,,,,,
region,candidate,retain as interpretable geographic back-off,All 21 levels are covered and functional rates...,Retain and add an LGA/region-grouped validatio...,Random validation can reward geographic memori...
